# 1. Purpose and scope

SCRUM-11 defines the finalized leakage-safe feature-engineering contract for Favorita forecasting. It translates the source-faithful sparse-data policy established by Notebooks 05 and 06 into approved, forbidden, and deferred feature rules without building a feature dataset.

This notebook is policy-only. It does not scan the 125,497,040-row cleaned dataset, materialize model-ready features, modify source or cleaned data, train or evaluate a model, or implement backtesting. The large feature build belongs to SCRUM-12; exact evaluation implementation belongs to SCRUM-13.

## 2. Stage boundaries and preserved lineage

- The governed source for later feature materialization is the validated cleaned artifact at `data/processed/favorita_cleaned/favorita_cleaned.parquet`.
- Only observed source-derived rows exist in that artifact; missing `(date, store_nbr, item_nbr)` rows remain semantically ambiguous.
- An absent store-item-date row does **not** prove zero demand.
- SCRUM-11 does not densify the historical panel, create synthetic rows, infer zero sales, impute values, or alter earthquake-period observations or metadata.
- Notebook 07 defines policy only. SCRUM-12 must implement the feature materialization contract, and SCRUM-13 must implement chronological evaluation details.

## 3. Forecasting contract

| Contract element | Final decision |
|---|---|
| Target | `unit_sales` |
| Forecast origin | End of calendar day `t` |
| Forecast dates | Daily targets from `t+1` through `t+14` |
| Output shape | One daily forecast for every horizon; never one combined 14-day total |
| Model design | Use a direct horizon-aware global forecasting design |
| Horizon field | `forecast_horizon` with integer values `1` through `14` |
| Prediction feedback | Forbidden: earlier-horizon predictions must not become later-horizon inputs |

Each model input row must also retain `forecast_origin` and `forecast_date` so the cutoff and target date are auditable. Recursive prediction feedback is not part of the approved design.

## 4. Governing leakage rule

> A feature is allowed only if the same value, with the same meaning, can be obtained or deterministically reconstructed using information available at the forecast origin.

The existence of future actual values in the historical merged or cleaned dataset does not make them valid model inputs. Availability is evaluated relative to forecast origin `t`, not relative to what is visible during retrospective notebook execution.

| Availability class | Meaning | Policy |
|---|---|---|
| Origin-bounded historical | Observed or derivable using information available at or before `t` | Allowed when its cutoff logic is explicit |
| Future-known | Planned or published value for target date `t+h` that was available at `t` | Conditionally allowed with availability evidence |
| Static | Store/item attribute whose semantics are stable and available at `t` | Allowed with identical joins in training and inference |
| Future actual | Value observed only after `t` | Forbidden |

## 5. Approved historical sales features

All sales features are calculated independently for each `(store_nbr, item_nbr)` series, anchored to forecast origin `t`, and reused for every direct horizon. Only actual sales observations available at or before `t` may contribute. No lag lookup or rolling window may cross the forecast origin.

| Approved feature | Semantic type | Availability | Policy |
|---|---|---|---|
| `sales_lag_1` | numeric | origin-bounded historical | Exact historical lag; never substitute zero for an absent row |
| `sales_lag_7` | numeric | origin-bounded historical | Same series and origin cutoff |
| `sales_lag_14` | numeric | origin-bounded historical | Same series and origin cutoff |
| `sales_lag_28` | numeric | origin-bounded historical | Same series and origin cutoff |
| `sales_rolling_mean_7` | numeric | origin-bounded historical | Window contains only eligible observations at or before `t` |
| `sales_rolling_mean_14` | numeric | origin-bounded historical | Window contains only eligible observations at or before `t` |
| `sales_rolling_mean_28` | numeric | origin-bounded historical | Window contains only eligible observations at or before `t` |
| `sales_rolling_std_7` | numeric | origin-bounded historical | Window contains only eligible observations at or before `t` |
| `sales_rolling_std_28` | numeric | origin-bounded historical | Window contains only eligible observations at or before `t` |

Do not materialize a zero merely because an exact lag date is absent, and do not silently densify the historical panel. Minimum-history, cold-start, exact date alignment, and sparse-window materialization behavior are deferred to SCRUM-12. `sales_change_*` features are not approved initially.

## 6. Approved future-known calendar features

Calendar features are deterministic properties of target date `t+h` and may be derived without observing future outcomes.

| Feature | Semantic type | Availability class |
|---|---|---|
| `day_of_week` | categorical or bounded integer calendar field | future-known deterministic |
| `day_of_month` | bounded integer calendar field | future-known deterministic |
| `week_of_year` | bounded integer calendar field | future-known deterministic |
| `month` | categorical or bounded integer calendar field | future-known deterministic |
| `quarter` | categorical or bounded integer calendar field | future-known deterministic |
| `is_weekend` | Boolean | future-known deterministic |

Sine/cosine cyclic transforms are not approved initially.

## 7. Promotion policy

`onpromotion` is a future-known **nullable Boolean** feature only when the planned value for target date `t+h` was available at forecast origin `t`.

- Preserve missing historical promotion values as unknown.
- Do not interpret missing `onpromotion` as `False`.
- Do not use a retrospectively observed target-date promotion value unless its planned value was genuinely available at `t` with the same meaning.
- Promotion-history lag features are not approved initially.
- SCRUM-12 must make unavailable planned promotions remain explicitly unknown rather than silently changing their semantics.

## 8. Approved static features and semantic types

| Feature | Required semantic type | Policy |
|---|---|---|
| `store_nbr` | categorical identifier | Never treat code magnitude as continuous |
| `item_nbr` | categorical identifier | Never treat code magnitude as continuous |
| `family` | categorical | Preserve source category lineage |
| `class` | categorical | Categorical even though physically numeric |
| `perishable` | binary | Preserve binary meaning |
| `city` | categorical | Use the same static join at training and inference |
| `state` | categorical | Use the same static join at training and inference |
| `store_type` | categorical | Preserve source category lineage |
| `cluster` | categorical identifier | Retain original integer code; do not convert to arbitrary letters or continuous magnitude |

Original codes remain available for lineage. `id` is excluded from model features because it is only a row identifier.

## 9. Historical transactions policy

Future target-date actual `transactions` values are forbidden. Only origin-bounded, store-level transaction history is approved.

| Approved feature | Semantic type | Cutoff policy |
|---|---|---|
| `transactions_at_origin` | nullable numeric | Actual store transactions known at or before end of `t` |
| `transactions_mean_7d` | nullable numeric | Historical store-level window ending no later than `t` |
| `transactions_mean_14d` | nullable numeric | Historical store-level window ending no later than `t` |
| `transactions_lag_7` | nullable numeric | Historical store-level lag relative to the origin |
| `transactions_lag_14` | nullable numeric | Historical store-level lag relative to the origin |

Preserve missing transaction values. Do not zero-fill, forward-fill, backward-fill, interpolate, or substitute future actual transactions. Exact date alignment and minimum-observation behavior are SCRUM-12 decisions.

## 10. Historical oil-movement policy

Raw `dcoilwtico` is excluded from model inputs, and all actual oil prices after forecast origin `t` are forbidden. Only relative historical movement features computed entirely from information available at or before `t` may be considered.

| Allowed feature family | Semantic type | Availability |
|---|---|---|
| `oil_pct_change_1d` | nullable numeric | origin-bounded historical |
| `oil_pct_change_7d` | nullable numeric | origin-bounded historical |
| `oil_rolling_change_7d` | nullable numeric | origin-bounded historical |
| `oil_rolling_volatility_7d` | nullable numeric | origin-bounded historical |

Do not interpolate or backfill missing source oil prices. Before materialization, SCRUM-12 must finalize exact formulas, date alignment, minimum-observation behavior, denominator-zero handling, and missing-value behavior.

## 11. Holiday and extraordinary-event policy

The following target-date fields are conditionally allowed only when they represent planned or published information for `t+h` that was available at forecast origin `t`.

| Feature | Semantic type | Policy |
|---|---|---|
| `is_holiday` | Boolean | Future-known only; broader than a strict public-holiday-only flag |
| `holiday_type` | categorical | Future-known only |
| `holiday_locale` | categorical | Future-known only |
| `holiday_transferred` | Boolean | Future-known only; renamed/merged source `transferred` concept |
| `holiday_event_count` | numeric count | Future-known only |

Lineage matters: `is_holiday` and `holiday_event_count` were engineered during merging, and the merged `is_holiday` field represents broader applicable calendar-event context. `holiday_description` is excluded.

All earthquake-related forecasting features, earthquake flags, days-since-earthquake features, and future earthquake assumptions are excluded. Genuine earthquake-period sales rows and source metadata remain preserved in the cleaned dataset; excluding features must not delete, relabel, or alter those historical rows.

## 12. Explicit forbidden-feature and transformation list

The following are forbidden:

- `id`;
- future actual `unit_sales`;
- future actual `transactions`;
- raw `dcoilwtico` and future actual oil prices;
- `holiday_description`;
- earthquake-derived features, earthquake flags, days-since-earthquake features, and future earthquake assumptions;
- recursive predicted-value feedback or earlier-horizon predictions used as later-horizon inputs;
- any lag or rolling calculation crossing forecast origin `t`;
- any preprocessing, encoding, imputation, aggregation, threshold, feature selection, or vocabulary fitted using validation/test information;
- synthetic zero-demand values inferred from missing rows;
- silent historical-panel densification;
- any training feature that cannot be recreated with identical semantics during inference.

`sales_change_*`, promotion-history lags, and calendar sine/cosine transforms are also outside the initial approved feature set.

## 13. Train/inference parity contract

Engineered training, validation, and inference inputs must use:

- identical ordered feature columns;
- identical semantic types and categorical treatment;
- identical null/unknown representation;
- identical static joins;
- identical target-date, lag, rolling-window, and forecast-origin cutoff logic;
- identical holiday eligibility and future-known availability rules.

Training-fitted encoders or vocabularies must be reused unchanged during validation and inference. They must never be refitted on validation, test, or inference data. The inference matrix must not contain the target.

Every engineered row must record `forecast_origin`, `forecast_date`, and `forecast_horizon` for auditability. A parity failure must block training or inference.

## 14. Chronological evaluation intent

SCRUM-13 will implement leakage-safe expanding-window or walk-forward evaluation with 14-day forecast windows. Random train/test splits and random 20% hold-outs are forbidden.

For each fold:

1. Train only through the fold's forecast origin.
2. Hide the following 14 days of actual `unit_sales` from feature construction.
3. Predict each of those 14 daily horizons.
4. Use the hidden actual sales only afterward for evaluation.

After the simulated clock advances, a previous fold's test period may become historical training data in a later fold. That is valid walk-forward behavior because the observations are then in the simulated past.

The original Favorita `test.csv` is not the primary evaluation dataset because it lacks `unit_sales`. Exact fold dates, metrics, weighting, fold count, and final hold-out policy remain SCRUM-13 work.

## 15. Approved-now summary

| Feature group | Approved now | Controlling condition |
|---|---|---|
| Forecast identifiers | `forecast_origin`, `forecast_date`, `forecast_horizon` | Required for every direct-horizon row |
| Sales history | Listed lags, rolling means, and rolling standard deviations | Same series; actual observations at or before `t` only |
| Calendar | Six listed target-date calendar fields | Deterministic from `t+h` |
| Promotion | Nullable `onpromotion` | Planned `t+h` value demonstrably available at `t` |
| Static | Listed store/item descriptors | Preserve categorical semantics and stable joins |
| Transactions | Five listed store-history features | Origin-bounded; preserve missingness |
| Oil | Four listed relative movement features | Origin-bounded; never raw or future actual prices |
| Holiday | Five listed fields | Planned/published and known at `t` |

Approval is conditional on the governing leakage rule and train/inference parity contract; a listed name is not permission to use retrospectively observed future values.

## 16. Assumptions requiring implementation evidence

- “Known at forecast origin” means available by the end-of-day `t` cutoff through a reproducible operational or historical-as-of source—not merely present in the final merged table.
- Static attributes are assumed usable only while their effective values and join semantics are valid at the simulated origin.
- Calendar fields are deterministically reconstructable from `forecast_date`.
- Planned promotion and holiday fields may be unavailable for some origins or horizons; conditional eligibility does not assert universal historical availability.
- Sparse-row semantics remain unchanged until SCRUM-12 explicitly defines materialization behavior.

If any assumption cannot be supported with identical training and inference semantics, the affected feature must be excluded or represented as unknown.

## 17. Details deferred to SCRUM-12

SCRUM-12 must implement and validate the large feature build, including:

- exact lag-date and rolling-window formulas under sparse observations;
- minimum-history and cold-start behavior;
- feature-row materialization without silent zero inference or densification;
- exact transaction and oil date alignment;
- oil denominator-zero, minimum-observation, and missing-value behavior;
- as-of availability checks for planned promotions and holiday metadata;
- static-attribute join rules and semantic-type enforcement;
- ordered feature schema, null representation, reusable encoder/vocabulary artifacts, and parity assertions;
- bounded-memory processing, artifact lineage, and executable quality checks.

These implementation details must not weaken the governing forecast-origin cutoff.

## 18. Evaluation details deferred to SCRUM-13

SCRUM-13 must define and execute exact walk-forward folds, forecast-origin dates, metrics, weighting, fold aggregation, final hold-out policy, and evaluation reporting. It must enforce the 14-day hidden-outcome procedure and prevent any validation/test information from influencing feature construction, fitting, preprocessing, thresholds, encoders, or vocabularies.

## 19. Final SCRUM-11 policy conclusion

**SCRUM-11 is finalized as a leakage-safe, direct horizon-aware feature policy.** Every allowed feature is either origin-bounded historical, deterministically future-known, conditionally planned/published and known at the origin, or static with preserved semantics. Future actuals, recursive feedback, target leakage, random evaluation splits, silent densification, and synthetic zero-demand inference are forbidden.

### Ready for SCRUM-12

- [x] Forecast origin, 14 daily horizons, target, and direct design defined.
- [x] Approved feature families and semantic types documented.
- [x] Sparse-row and missing-value policies preserved.
- [x] Explicit forbidden-feature list documented.
- [x] Train/inference parity is a blocking contract.
- [x] Chronological evaluation intent documented for SCRUM-13.
- [x] Implementation details are explicitly deferred to SCRUM-12.

No feature artifact, model, synthetic row, or evaluation result is created by this notebook.